# Stage 10a — XAI Evaluation: Stage 29 (L3+L4 with skip)

**Goal:** Compute AP, Faithfulness, Stability for `proto_seg_ct_l3l4_warmstart.pth`  
**Why:** This is the missing half of the core skip vs no-skip comparison table (Two-Barrier Framework)

Also covers **G2**: aggregate `xai_summary_9a.csv` from existing v9 component files.

| Model | Skip | Levels | Dice | Purity | AP | Faithfulness | Stability |
|-------|------|--------|------|--------|-----|-------------|----------|
| Stage 29 | ✅ | L3+L4 | 0.866 | 0.649 | ??? | ??? | ??? |
| 9b | ❌ | L3+L4 | 0.559 | 0.686 | 0.301 | 0.035 | 11.94 |
| 9a | ❌ | L4 | 0.606 | 0.689 | 0.312 | 0.012 | 10.92 |

In [1]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import torch
import torch.nn as nn
import torch.utils.data as tud
import numpy as np
import pandas as pd
from pathlib import Path

from src.data.mmwhs_dataset import MMWHSSliceDataset, LABEL_NAMES, NUM_CLASSES
from src.models.proto_seg_net import ProtoSegNet
from src.metrics.proto_quality import (
    compute_purity, compute_per_level_ap, compute_compactness,
    compute_level_dominance, compute_effective_quality,
)
from src.metrics.faithfulness import faithfulness_patient
from src.metrics.stability import stability_patient

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR  = 'data/pack/processed_data'
CKPT_DIR  = Path('checkpoints')
V9_DIR    = Path('results/v9')
OUT_DIR   = Path('results/v10')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODALITY    = 'ct'
PROTO_LEVELS = [3, 4]

print(f'Working dir : {os.getcwd()}')
print(f'Device      : {DEVICE}')
print(f'Output dir  : {OUT_DIR}')

Working dir : /Users/amo/programData/cardiac-proto-segmentation
Device      : mps
Output dir  : results/v10


## 1. Load Stage 29 (ProtoSegNet L3+L4 with skip)

In [2]:
CKPT_PATH = CKPT_DIR / 'proto_seg_ct_l3l4_warmstart.pth'
PROJ_PATH = CKPT_DIR / 'projected_prototypes_ct_l3l4_warmstart.pt'

ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
print(f'Checkpoint  : epoch={ckpt["epoch"]}  best_val={ckpt["best_val_dice"]:.4f}')
print(f'Proto levels: {ckpt.get("proto_levels", PROTO_LEVELS)}')

model = ProtoSegNet(
    n_classes=NUM_CLASSES,
    proto_levels=PROTO_LEVELS,
    use_level_attention=ckpt.get('use_level_attention', False),
).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

# Load projected prototypes
if PROJ_PATH.exists():
    proj = torch.load(PROJ_PATH, map_location='cpu', weights_only=True)
    for level, proto_data in proj['proto_state'].items():
        model.proto_layers[str(level)].prototypes.data.copy_(proto_data)
    print(f'Projected prototypes loaded from {PROJ_PATH.name}')
else:
    print('WARNING: no projection file — using trained (unprojected) prototypes')

model.eval()
print('Model ready.')

Checkpoint  : epoch=10  best_val=0.8215
Proto levels: [3, 4]
Projected prototypes loaded from projected_prototypes_ct_l3l4_warmstart.pt
Model ready.


## 2. Wrapper for 3-output interface

`faithfulness_patient` and `stability_patient` call `model(x)` and unpack 3 values  
`(logits, heatmaps, w)` — matching ProtoSegNetV2's interface.  
ProtoSegNet returns only 2: `(logits, heatmaps)`.  
A thin wrapper adds a dummy weight tensor.

In [3]:
class ProtoSegNetV2Adapter(nn.Module):
    """Wraps ProtoSegNet (2-output) to expose ProtoSegNetV2's 3-output interface."""
    def __init__(self, base: ProtoSegNet):
        super().__init__()
        self.base = base

    def forward(self, x, **kwargs):
        logits, heatmaps = self.base(x)
        B = x.size(0)
        # dummy weight: uniform over proto_levels (not used by faithfulness/stability)
        n_levels = len(self.base.proto_levels)
        w = torch.full((B, n_levels), 1.0 / n_levels, device=x.device)
        return logits, heatmaps, w

adapted_model = ProtoSegNetV2Adapter(model).to(DEVICE)
adapted_model.eval()

# Smoke test
with torch.no_grad():
    x_test = torch.randn(2, 1, 256, 256).to(DEVICE)
    logits, hm, w = adapted_model(x_test)
print(f'Adapter output: logits={tuple(logits.shape)}  w={tuple(w.shape)}')
for l, A in hm.items():
    print(f'  hm[{l}] : {tuple(A.shape)}')

Adapter output: logits=(2, 8, 256, 256)  w=(2, 2)
  hm[3] : (2, 8, 2, 32, 32)
  hm[4] : (2, 8, 2, 16, 16)


## 3. Data Loaders

In [4]:
test_ds  = MMWHSSliceDataset(DATA_DIR, MODALITY, 'test',  augment=False, preload=True)
train_ds = MMWHSSliceDataset(DATA_DIR, MODALITY, 'train', augment=False, preload=True)
test_loader  = tud.DataLoader(test_ds,  batch_size=16, shuffle=False)
train_loader = tud.DataLoader(train_ds, batch_size=16, shuffle=False)
print(f'Test : {len(test_ds)} slices')
print(f'Train: {len(train_ds)} slices')

Test : 484 slices
Train: 3389 slices


## 4. Prototype Quality (Purity, AP, Compactness, Dominance)

These metrics call `model(x)` and expect 2 outputs — use `model` (not adapter).

In [5]:
print('Computing purity (train set)…')
purity_df  = compute_purity(model, train_loader)

print('Computing AP (test set)…')
ap_df      = compute_per_level_ap(model, test_loader)

print('Computing compactness (test set)…')
compact_df = compute_compactness(model, test_loader)

print('Computing level dominance (test set)…')
dom_df     = compute_level_dominance(model, test_loader)

print('Computing effective quality…')
eq_df      = compute_effective_quality(purity_df, ap_df, compact_df, dom_df)

# Save
purity_df.to_csv(OUT_DIR / 'xai_purity_stage29.csv',            index=False)
ap_df.to_csv    (OUT_DIR / 'xai_ap_stage29.csv',                 index=False)
compact_df.to_csv(OUT_DIR / 'xai_compactness_stage29.csv',       index=False)
eq_df.to_csv    (OUT_DIR / 'xai_effective_quality_stage29.csv',  index=False)

print('\n─── Per-level Purity ───')
print(purity_df.groupby('level')[['purity']].mean().round(3).to_string())
print('\n─── Per-level AP ───')
print(ap_df.groupby('level')[['ap']].mean().round(3).to_string())
print('\n─── Effective quality ───')
print(eq_df.round(3).to_string(index=False))

Computing purity (train set)…


Computing AP (test set)…


Computing compactness (test set)…


Computing level dominance (test set)…


Computing effective quality…

─── Per-level Purity ───
       purity
level        
3       0.001
4       0.070

─── Per-level AP ───
          ap
level       
3      0.032
4      0.020

─── Effective quality ───
 weight_l3  purity_l3  ap_l3  compact_l3  weight_l4  purity_l4  ap_l4  compact_l4  effective_purity  effective_ap  effective_compactness
     0.549      0.001  0.032       0.757      0.451       0.07   0.02       0.738             0.032         0.026                  0.748


## 5. Faithfulness & Stability

These metrics expect 3-output interface — use `adapted_model`.

In [6]:
imgs_test = torch.stack([test_ds[i]['image'] for i in range(len(test_ds))])
print(f'Test images: {tuple(imgs_test.shape)}')

print('\nComputing Faithfulness (50 slices)…')
faith = faithfulness_patient(adapted_model, imgs_test, DEVICE, max_slices=50)
print(f'  Faithfulness : {faith["faithfulness"]:.4f}  (std {faith["faithfulness_std"]:.4f})')

print('\nComputing Stability (50 slices)…')
stab = stability_patient(adapted_model, imgs_test, DEVICE, max_slices=50)
print(f'  Stability    : {stab["stability"]:.4f}  (std {stab["stability_std"]:.4f})')

pd.DataFrame([faith]).to_csv(OUT_DIR / 'xai_faithfulness_stage29.csv', index=False)
pd.DataFrame([stab ]).to_csv(OUT_DIR / 'xai_stability_stage29.csv',    index=False)
print('Saved.')

Test images: (484, 1, 256, 256)

Computing Faithfulness (50 slices)…


  Faithfulness : 0.0689  (std 0.0467)

Computing Stability (50 slices)…


  Stability    : 3.3816  (std 0.5948)
Saved.


## 6. Summary Table — Stage 29

In [7]:
eff_ap   = float(eq_df['effective_ap'].iloc[0])
eff_pur  = float(eq_df['effective_purity'].iloc[0])
eff_comp = float(eq_df['effective_compactness'].iloc[0])
faith_v  = faith['faithfulness']
stab_v   = stab['stability']
val_dice = ckpt['best_val_dice']   # 0.8656 (val), 3D test = 0.8656

rows_29 = [
    ('Val Dice',           val_dice, None, None, '—'),
    ('Effective Purity',   eff_pur,  None, None, '—'),
    ('Effective AP',       eff_ap,   0.15, 0.25, '✅' if eff_ap  >= 0.15 else '❌'),
    ('Effective Compact.', eff_comp, None, None, '—'),
    ('Faithfulness',       faith_v,  0.15, 0.30, '✅' if faith_v >= 0.15 else '❌'),
    ('Stability',          stab_v,   None, 2.00, '✅' if stab_v  <= 2.00 else '❌'),
]
summary_29 = pd.DataFrame(rows_29, columns=['Metric', 'Value', 'Min gate', 'Target', 'Pass'])
summary_29.to_csv(OUT_DIR / 'xai_summary_stage29.csv', index=False)

print('Stage 29 (L3+L4, WITH skip) — XAI Summary')
print('=' * 55)
print(summary_29.to_string(index=False))

Stage 29 (L3+L4, WITH skip) — XAI Summary
            Metric    Value  Min gate  Target Pass
          Val Dice 0.821452       NaN     NaN    —
  Effective Purity 0.032362       NaN     NaN    —
      Effective AP 0.026332      0.15    0.25    ❌
Effective Compact. 0.748157       NaN     NaN    —
      Faithfulness 0.068851      0.15    0.30    ❌
         Stability 3.381648       NaN    2.00    ❌


## G2 — Aggregate xai_summary_9a.csv

All component files exist; only the summary was missing.

In [8]:
eq_9a    = pd.read_csv(V9_DIR / 'xai_effective_quality_9a.csv')
faith_9a = pd.read_csv(V9_DIR / 'xai_faithfulness_9a.csv')
stab_9a  = pd.read_csv(V9_DIR / 'xai_stability_9a.csv')

rows_9a = [
    ('Val Dice',           0.6055,                           None, None, '—'),
    ('Effective Purity',   float(eq_9a['effective_purity'].iloc[0]),  None, None, '—'),
    ('Effective AP',       float(eq_9a['effective_ap'].iloc[0]),      0.15, 0.25,
     '✅' if float(eq_9a['effective_ap'].iloc[0]) >= 0.15 else '❌'),
    ('Effective Compact.', float(eq_9a['effective_compactness'].iloc[0]), None, None, '—'),
    ('Faithfulness',       float(faith_9a['faithfulness'].iloc[0]),   0.15, 0.30,
     '✅' if float(faith_9a['faithfulness'].iloc[0]) >= 0.15 else '❌'),
    ('Stability',          float(stab_9a['stability'].iloc[0]),       None, 2.00,
     '✅' if float(stab_9a['stability'].iloc[0]) <= 2.00 else '❌'),
]
summary_9a = pd.DataFrame(rows_9a, columns=['Metric', 'Value', 'Min gate', 'Target', 'Pass'])
summary_9a.to_csv(V9_DIR / 'xai_summary_9a.csv', index=False)

print('9a (L4, NO skip) — XAI Summary')
print('=' * 55)
print(summary_9a.to_string(index=False))
print(f'\nSaved to {V9_DIR}/xai_summary_9a.csv')

9a (L4, NO skip) — XAI Summary
            Metric     Value  Min gate  Target Pass
          Val Dice  0.605500       NaN     NaN    —
  Effective Purity  0.678571       NaN     NaN    —
      Effective AP  0.312173      0.15    0.25    ✅
Effective Compact.  0.035526       NaN     NaN    —
      Faithfulness  0.012267      0.15    0.30    ❌
         Stability 10.920775       NaN    2.00    ❌

Saved to results/v9/xai_summary_9a.csv


## 7. Two-Barrier Comparison Table

Core table for the v10 paper narrative.

In [9]:
# Load 9b summary
s9b = pd.read_csv(V9_DIR / 'xai_summary_9b.csv').set_index('Metric')['Value']
s9a = summary_9a.set_index('Metric')['Value']
s29 = summary_29.set_index('Metric')['Value']

def fmt(v):
    try: return f'{float(v):.3f}'
    except: return str(v)

metrics = ['Val Dice', 'Effective Purity', 'Effective AP', 'Faithfulness', 'Stability']

print('=' * 75)
print(f'  Two-Barrier Framework — Core Comparison Table (CT)')
print('=' * 75)
print(f'  {"Metric":<22}  {"Stage 29":>12}  {"9b":>12}  {"9a":>12}  {"Δ (29→9b)":>12}')
print(f'  {"":.<22}  {"L3+L4 skip":>12}  {"L3+L4 noskip":>12}  {"L4 noskip":>12}  {"":>12}')
print('  ' + '─' * 71)
for m in metrics:
    v29 = float(s29.get(m, float('nan')))
    v9b = float(s9b.get(m, float('nan')))
    v9a = float(s9a.get(m, float('nan')))
    delta = v9b - v29
    arrow = '↑' if delta > 0 else '↓'
    print(f'  {m:<22}  {v29:>12.3f}  {v9b:>12.3f}  {v9a:>12.3f}  {delta:>+11.3f}{arrow}')
print('=' * 75)
print()
print('Interpretation:')
print('  Barrier 1 (Bypass): Skip → higher AP. Removing skip means AP ↑ (9b > Stage 29)')
print('  Barrier 2 (Resolution): No-skip has WORSE Faithfulness/Stability (coarser feature maps)')
print('  Skip connection is NOT the cause of poor Faithfulness — resolution is.')

  Two-Barrier Framework — Core Comparison Table (CT)
  Metric                      Stage 29            9b            9a     Δ (29→9b)
  ......................    L3+L4 skip  L3+L4 noskip     L4 noskip              
  ───────────────────────────────────────────────────────────────────────
  Val Dice                       0.821         0.559         0.606       -0.263↓
  Effective Purity               0.032         0.686         0.679       +0.654↑
  Effective AP                   0.026         0.301         0.312       +0.275↑
  Faithfulness                   0.069         0.035         0.012       -0.034↓
  Stability                      3.382        11.936        10.921       +8.555↑

Interpretation:
  Barrier 1 (Bypass): Skip → higher AP. Removing skip means AP ↑ (9b > Stage 29)
  Barrier 2 (Resolution): No-skip has WORSE Faithfulness/Stability (coarser feature maps)
  Skip connection is NOT the cause of poor Faithfulness — resolution is.


## G6 — Per-class Dice for Stage 29

Extract from the last val epoch logged in the training curve.

In [10]:
# Try to find Stage 29 training curve
v8_curves = [
    Path('results/v8/train_curve_proto_ct_l3l4_warmstart.csv'),
    Path('results/v7/train_curve_proto_ct_l3l4_warmstart.csv'),
    Path('results/train_curve_proto_ct_l3l4_warmstart.csv'),
]
curve_found = None
for p in v8_curves:
    if p.exists():
        curve_found = p
        break

FG_NAMES = [LABEL_NAMES[c] for c in range(1, NUM_CLASSES)]

if curve_found:
    curve = pd.read_csv(curve_found)
    # Last row with val data
    val_rows = curve.dropna(subset=['val_mean_fg_dice'])
    best_row = val_rows.loc[val_rows['val_mean_fg_dice'].idxmax()]
    print(f'Stage 29 best val epoch: {int(best_row["epoch"])}  Dice: {best_row["val_mean_fg_dice"]:.4f}')
    print()
    print('Per-class Dice (Stage 29):')
    for name in FG_NAMES:
        col = f'val_dice_{name}'
        if col in best_row:
            print(f'  {name:<12}: {float(best_row[col]):.4f}')
else:
    print('Training curve for Stage 29 not found in standard locations.')
    print('Computing from checkpoint via validation pass instead…')
    
    from src.data.mmwhs_dataset import make_dataloaders
    from src.metrics.dice import dice_per_class, mean_foreground_dice
    loaders = make_dataloaders(DATA_DIR, MODALITY, batch_size=16)
    val_loader = loaders['val']
    
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits, _ = model(batch['image'].to(DEVICE))
            all_logits.append(logits.cpu())
            all_labels.append(batch['label'])
    val_dice = dice_per_class(torch.cat(all_logits), torch.cat(all_labels))
    val_mean = mean_foreground_dice(val_dice)
    print(f'Val Dice (checkpoint): {val_mean:.4f}')
    print()
    print('Per-class Dice (Stage 29):')
    for name in FG_NAMES:
        print(f'  {name:<12}: {val_dice.get(name, float("nan")):.4f}')

Training curve for Stage 29 not found in standard locations.
Computing from checkpoint via validation pass instead…


Val Dice (checkpoint): 0.8164

Per-class Dice (Stage 29):
  LV          : 0.7238
  RV          : 0.8238
  LA          : 0.9112
  RA          : 0.7936
  Myocardium  : 0.8344
  Aorta       : 0.9074
  PA          : 0.7204
